# 06 - Smoothed latent vectors

The 7-day baseline clusters the backward 7-day mean of the daily mean and standard deviation of the price. The Temporal VAE latents were never smoothed the same way, so this notebook compares both with the same help. No model is trained: it reads the latents saved by notebook 03.

For every trained model in `studies` and every k in `k_values`, k-means runs on these inputs:
- `level 7d`: the 7-day baseline (daily mean and standard deviation, backward 7-day mean, z-scored on training days).
- `latents + level`: the latents plus the daily mean and standard deviation, as in notebook 03.
- `latents 7d + level 7d`: the latents and the daily mean and standard deviation, all replaced by their backward 7-day mean (the day and the 6 days before it; the first 6 days average the days available).

The two inputs with latents are run twice: with the latents as saved (`raw`) and with each latent dim z-scored on training days (`scaled`), as the price is. In notebook 03 the latents enter raw, so how much they weigh against the price depends on their spread.

As in notebook 03, k-means is fitted on training days, assigns every day, and regimes are numbered by ascending mean real price. Results are saved in `experiments/temporal_vae/kmeans/latent_smoothing/`. The size-aware metrics of the most expensive regime (`src/metrics.py`) are computed for every input too and saved as `metrics/smoothing_metrics.csv`, which `scripts/06_regime_metrics.py` adds to `metrics/regime_metrics.csv`.

<d.benavidess@uniandes.edu.co>

In [8]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import load_context, chronological_split
from src.clustering import level_columns, fit_clusters, order_regimes, summary_metrics
from src.runs import nino_episodes, nino_share_in_top
from src.metrics import days_from_context, regime_metrics
from src import experiments as exp

In [9]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`studies` are the study folders of notebook 03 whose trained models are read. Each trained model is read once, from the folder named by `trained_in`, so its `artifacts/latents.npy` must be on this computer. `window` is the length of the backward mean in days.

In [10]:
scalogram_set = "real_price_log"
studies = ["temporal_vae/kmeans/hyperparameters", "temporal_vae/kmeans/lam_beta_grid"]
k_values = [3, 4, 5]
window = 7
train_ratio, val_ratio = 0.6, 0.2
out_dir = exp.EXPERIMENTS_DIR / "temporal_vae" / "kmeans" / "latent_smoothing"

## 1. Data and trained models

In [11]:
metadata = pd.read_csv(Path("../data/processed/scalograms") / scalogram_set / "scalogram_metadata.csv",
                       parse_dates=["start_datetime", "end_datetime"])
context = load_context(Path("../data/processed"), metadata)
train_idx, _, _ = chronological_split(len(metadata), train_ratio, val_ratio)
price = context["precio_real_cop_kwh"].to_numpy()
days = days_from_context(context)

models = {}
for study in studies:
    for path in sorted((exp.EXPERIMENTS_DIR / study).glob("*/config.json")):
        cfg = json.loads(path.read_text())
        if cfg.get("scalogram_set") == scalogram_set:
            models.setdefault(cfg["trained_in"], {"study": cfg["study"], "lam": cfg["lam"], "beta": cfg["beta"],
                                                  "seed": cfg["seed"]})
models = pd.DataFrame.from_dict(models, orient="index").rename_axis("trained_in").reset_index()
models

,trained_in,study,lam,beta,seed
0,temporal_vae/kmeans/hyperparameters/001_real_p...,hyperparameters,100.0,1.00,42
1,temporal_vae/kmeans/hyperparameters/006_real_p...,hyperparameters,100.0,1.00,43
2,temporal_vae/kmeans/hyperparameters/008_real_p...,hyperparameters,100.0,1.00,44
3,temporal_vae/kmeans/lam_beta_grid/001_real_pri...,lam_beta_grid,0.0,0.25,42
4,temporal_vae/kmeans/lam_beta_grid/004_real_pri...,lam_beta_grid,0.0,0.50,42
5,temporal_vae/kmeans/lam_beta_grid/007_real_pri...,lam_beta_grid,100.0,0.25,42
6,temporal_vae/kmeans/lam_beta_grid/010_real_pri...,lam_beta_grid,100.0,0.50,42
7,temporal_vae/kmeans/lam_beta_grid/013_real_pri...,lam_beta_grid,300.0,0.25,42
8,temporal_vae/kmeans/lam_beta_grid/016_real_pri...,lam_beta_grid,300.0,0.50,42


## 2. Clustering

`zscore` scales columns with the mean and standard deviation of the training days. `backward_mean` is the backward `window`-day mean of each column.

In [12]:
def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    return (x - x[train_idx].mean(axis=0)) / x[train_idx].std(axis=0)


def backward_mean(x):
    return pd.DataFrame(x).rolling(window, min_periods=1).mean().to_numpy()


level = metadata[list(level_columns(metadata))].to_numpy(dtype=np.float64)
level_day, level_7d = zscore(level), zscore(backward_mean(level))

rows, metric_rows = [], []
for _, model in models.iterrows():
    latents = np.load(exp.EXPERIMENTS_DIR / model["trained_in"] / "artifacts" / "latents.npy").astype(np.float64)
    latents_7d = backward_mean(latents)
    feature_sets = {
        ("level 7d", "-"): level_7d,
        ("latents + level", "raw"): np.hstack([latents, level_day]),
        ("latents + level", "scaled"): np.hstack([zscore(latents), level_day]),
        ("latents 7d + level 7d", "raw"): np.hstack([latents_7d, level_7d]),
        ("latents 7d + level 7d", "scaled"): np.hstack([zscore(latents_7d), level_7d]),
    }
    for (features, latent_scale), x in feature_sets.items():
        for k in k_values:
            labels = order_regimes(fit_clusters(x, train_idx, k, "kmeans", int(model["seed"]))[0], price)
            ctx = context.assign(regime=labels)
            m = summary_metrics(x, labels, seed=int(model["seed"]))
            rows.append({
                **model.to_dict(), "features": features, "latents": latent_scale, "k": k,
                "switches_per_year": m["switches_per_year"], "mean_run_days": m["mean_run_days"],
                **{f"nino_{name[:4]}_{name[7:9]}_in_top_pct": nino_share_in_top(ctx, years)
                   for name, years in nino_episodes.items()},
                "top_share_pct": 100 * (labels == k - 1).mean(),
            })
            smoothed = features != "latents + level"
            for view in ("daily", "7d"):
                metric_rows.append({
                    "model": "temporal_vae" if features != "level 7d" else "level_only", "study": model["study"],
                    "run_id": model["trained_in"], "lam": model["lam"], "beta": model["beta"], "seed": model["seed"],
                    "k": k, "features": features, "latents": latent_scale, "smoothed": smoothed,
                    **regime_metrics(days.assign(regime=labels), k, view), "matched": (view == "7d") == smoothed,
                })
results = pd.DataFrame(rows)
out_dir.mkdir(parents=True, exist_ok=True)
results.round(2).to_csv(out_dir / "smoothing_results.csv", index=False)
metrics_dir = exp.ROOT / "metrics"
metrics_dir.mkdir(exist_ok=True)
pd.DataFrame(metric_rows).round(3).to_csv(metrics_dir / "smoothing_metrics.csv", index=False)
print(f"{len(models)} models x {len(feature_sets)} inputs x {len(k_values)} k = {len(results)} rows")

9 models x 5 inputs x 3 k = 135 rows


## 3. Summary

Mean and standard deviation over seeds for every model configuration (study, lam, beta), input and k. With one seed the standard deviation is empty. Saved as `smoothing_summary.csv`.

In [13]:
columns = ["switches_per_year", "mean_run_days", "nino_2015_16_in_top_pct", "nino_2023_24_in_top_pct", "top_share_pct"]
group = ["k", "study", "lam", "beta", "features", "latents"]
summary = results.groupby(group, sort=False)[columns].agg(["mean", "std"]).round(1)
summary["seeds"] = results.groupby(group, sort=False).size()
summary = summary.sort_index(level=["k", "study", "lam", "beta"], sort_remaining=False)
summary.to_csv(out_dir / "smoothing_summary.csv")
summary

switches_per_year  \
                                                                        mean   
k study           lam   beta features              latents                     
3 hyperparameters 100.0 1.0  level 7d              -                    15.3   
                             latents + level       raw                  79.4   
                                                   scaled               52.9   
                             latents 7d + level 7d raw                  16.1   
                                                   scaled               13.9   
...                                                                      ...   
5 lam_beta_grid   300.0 0.5  level 7d              -                    25.6   
                             latents + level       raw                  78.6   
                                                   scaled               77.3   
                             latents 7d + level 7d raw                  23.7   
                                                   scaled               26.1   

                                                                mean_run_days  \
                                                            std          mean   
k study           lam   beta features              latents                      
3 hyperparameters 100.0 1.0  level 7d              -        0.0          23.8   
                             latents + level       raw      8.3           4.6   
                                                   scaled   2.1           6.9   
                             latents 7d + level 7d raw      0.5          22.7   
                                                   scaled   0.7          26.3   
...                                                         ...           ...   
5 lam_beta_grid   300.0 0.5  level 7d              -        NaN          14.2   
                             latents + level       raw      NaN           4.6   
                                                   scaled   NaN           4.7   
                             latents 7d + level 7d raw      NaN          15.4   
                                                   scaled   NaN          13.9   

                                                                 \
                                                            std   
k study           lam   beta features              latents        
3 hyperparameters 100.0 1.0  level 7d              -        0.1   
                             latents + level       raw      0.5   
                                                   scaled   0.3   
                             latents 7d + level 7d raw      0.7   
                                                   scaled   1.3   
...                                                         ...   
5 lam_beta_grid   300.0 0.5  level 7d              -        NaN   
                             latents + level       raw      NaN   
                                                   scaled   NaN   
                             latents 7d + level 7d raw      NaN   
                                                   scaled   NaN   

                                                           nino_2015_16_in_top_pct  \
                                                                              mean   
k study           lam   beta features              latents                           
3 hyperparameters 100.0 1.0  level 7d              -                          91.2   
                             latents + level       raw                        85.8   
                                                   scaled                     68.5   
                             latents 7d + level 7d raw                        90.5   
                                                   scaled                     65.0   
...                                                                            ...   
5 lam_beta_grid   300.0 0.5  level 7d              -                          41.0   
    

## 4. One table per k

Means over seeds only, for reading. `level 7d` is the same for every model, so it is shown once per k at the top.

In [14]:
for k in k_values:
    table = summary.xs(k, level="k").xs("mean", axis=1, level=1)
    base = table.xs("level 7d", level="features").iloc[[0]].reset_index(drop=True)
    print(f"k = {k}   level 7d: " + ", ".join(f"{c} {base[c].iloc[0]:.1f}" for c in columns))
    print(table.drop(index="level 7d", level="features").to_string(), end="\n\n")

k = 3   level 7d: switches_per_year 15.3, mean_run_days 23.8, nino_2015_16_in_top_pct 91.2, nino_2023_24_in_top_pct 90.4, top_share_pct 34.1
                                                          switches_per_year  mean_run_days  nino_2015_16_in_top_pct  nino_2023_24_in_top_pct  top_share_pct
study           lam   beta features              latents                                                                                                   
hyperparameters 100.0 1.00 latents + level       raw                   79.4            4.6                     85.8                     82.3           39.4
                                                 scaled                52.9            6.9                     68.5                     61.8           45.1
                           latents 7d + level 7d raw                   16.1           22.7                     90.5                     88.8           33.1
                                                 scaled                13.9    